# MOSAIC GBM: ECM composition across TIL zones

Regenerates Supp Fig 5c, Fig 5d, Fig 5e, Supp Fig 5d and 5e, and the matrisome gene table. TIL-high and TIL-low sections are assigned as in `fig5c_suppfig5ab_TIL_supplement.ipynb`, the table saved as TIL_groups.csv. Every analysis here compares TIL zones, so the sections without any TIL+ spot, which are TIL-low in the survival analysis, are left out, because border zones are defined around TIL+ spots. Expression comes from the log_norm_count layer of the Visium cohort loaded through gbmcollaboration.

**Produces**: Supp Fig 5c, Fig 5d, Fig 5e (10-gene collagen panel, 177/192 sections engaged > cold,
p = 1.66e-30), Supp Fig 5d–e and the Supp Table 4a gene table (which analyses use each of its 123
genes; CTSD, IGFBP7 and LGALS3 are used by none; IGFBP2 by the NicheNet pathway filter and the
matrisome list of the ECM–TIL-zone tables, `supptables_4a_ecm_til_zone_tables.ipynb`).

**Origin**: Spencer Watson, `ECM_supplement.ipynb` (November 2026, not kept in the repository), all
cells in his order.

**Compared with the published outputs** (tables of the published runs, checked after each panel):
TIL+, BZ1 and BZ2 match; the TIL− zone differs because the published runs kept the 23 sections
without a TIL+ spot as TIL-low (171 instead of 148). Fig 5e reproduces
`Final_Notebooks/collagen_engaged_vs_cold_stats.csv` (n = 192, 177 / 192, p = 1.66e-30). Fig 5d is
drawn with all section-zones (`MIN_ZONE_SPOTS = 0`); the ≥ 20-spot variant printed after it
reproduces `Final_Notebooks/focus_niches_significance.csv` (TIL+ ECM_0: * instead of ***), so the
published stars came from that filter.

In [ ]:
import gc
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import mannwhitneyu, ttest_ind, wilcoxon
from statsmodels.stats.multitest import multipletests

from gbmcollaboration import AllDataCenters
from gbmcollaboration.paper import config, output, visium
from gbmcollaboration.paper.output import scanpy_style
from gbmcollaboration.paper.til import til_groups, zones_with_groups

In [ ]:
# Config
OUT = output.setup("fig5de_suppfig5cde_ECM_supplement")
PATHS = {
    "til_zones": config.TIL_ZONES_CSV,
    "ecm": config.ECM_ANNOTATIONS_CSV,
    "supp_table_4a": config.EFS / "supplementary_tables/4a_ECM_genes.csv",
}
PUBLISHED = {  # tables written by the published runs, for the checks after each panel
    "suppfig5c": config.EFS / "Analysis_Repo/Notebooks/ECM_nhood_TIL_zone_binary_bubble.csv",
    "fig5d": config.EFS / "Final_Notebooks/patient_fracs_zone_ecm_group.csv",
    "fig5d_tests": config.EFS / "Final_Notebooks/focus_niches_significance.csv",
    "fig5e": config.EFS / "Final_Notebooks/collagen_engaged_vs_cold_stats.csv",
    "abs": config.EFS / "Final_Notebooks/matrisome_absolute_per_zone_category.csv",
    "frac": config.EFS / "Final_Notebooks/matrisome_compositional_per_zone_category.csv",
}
QUANTILE_CUTOFF = 0.75
ZONES = ["TIL+", "BZ1", "BZ2", "TIL-"]
ECM_KEY = "ECM_neighborhood"

# Brain matrisome panel by functional category, as used for ECM neighborhood clustering
CATEGORY_GENES = {
    "Collagens": ["COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2", "COL5A1", "COL5A2", "COL5A3", "COL6A1",
                  "COL6A2", "COL6A3", "COL7A1", "COL8A1", "COL11A1", "COL12A1", "COL14A1", "COL15A1",
                  "COL18A1", "COL10A1"],
    "ECM glycoproteins": ["FN1", "FBN1", "FBN2", "TNC", "TNR", "TNXB", "SPARC", "POSTN", "SPP1", "EFEMP1",
                          "LAMA1", "LAMA2", "LAMA4", "LAMA5", "LAMB1", "LAMB2", "LAMB3", "LAMC1", "LAMC3",
                          "THBS1", "THBS2", "THBS4", "VTN", "NID1", "NID2", "FBLN1", "FBLN2", "FBLN5", "MMRN2",
                          "VWA1", "TGFBI", "LTBP1", "CHI3L1", "LGALS1", "CCN1", "CCN2", "EMILIN1"],
    "Proteoglycans": ["VCAN", "ACAN", "BCAN", "NCAN", "DCN", "BGN", "LUM", "PRELP", "FMOD", "ASPN", "HSPG2",
                      "AGRN", "GPC1", "GPC2", "GPC3", "SDC1", "SDC4", "CSPG4", "CSPG5", "PTPRZ1", "HAPLN1",
                      "HAPLN2", "HAPLN3", "HAPLN4"],
    "Proteases": ["MMP2", "MMP7", "MMP9", "MMP11", "MMP12", "MMP13", "MMP14", "MMP16", "ADAMTS1", "ADAMTS4",
                  "ADAMTS5", "ADAM8", "HPSE", "PLAU", "PLAT", "CTSB"],
    "Protease inhibitors": ["TIMP1", "TIMP2", "TIMP3", "SERPINE1"],
    "Crosslinking / collagen maturation": ["LOX", "LOXL1", "LOXL2", "LOXL3", "LOXL4", "TGM2", "P4HA1", "P4HA2",
                                           "PLOD1", "PLOD2", "BMP1", "PCOLCE"],
    "Hyaluronan metabolism": ["HAS2", "HAS3", "HYAL2"],
    "Other secreted ECM factors": ["IGFBP5", "SRGN"],
}
CATEGORY_ORDER = list(CATEGORY_GENES)
COLLAGEN_PANEL = ["COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2", "COL5A1", "COL5A2", "COL6A1", "COL6A2", "COL6A3"]

# Matrisome gene lists of the gene table
SUMMARY_LIST_119 = [
    "COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2", "COL5A1", "COL5A2", "COL5A3", "COL6A1", "COL6A2", "COL6A3",
    "COL7A1", "COL8A1", "COL10A1", "COL11A1", "COL12A1", "COL14A1", "COL15A1", "COL16A1", "COL18A1",
    "FN1", "FBN1", "FBN2", "TNC", "TNR", "TNXB", "SPARC", "POSTN", "SPP1", "EFEMP1", "LAMA1", "LAMA2", "LAMA3",
    "LAMA4", "LAMA5", "LAMB1", "LAMB2", "LAMB3", "LAMC1", "LAMC2", "LAMC3", "THBS1", "THBS2", "THBS4", "VTN",
    "NID1", "NID2", "FBLN1", "FBLN2", "FBLN5", "MMRN2", "VWA1", "TGFBI", "LTBP1",
    "VCAN", "ACAN", "BCAN", "NCAN", "DCN", "BGN", "LUM", "PRELP", "FMOD", "ASPN", "HSPG2", "AGRN", "GPC1", "GPC2",
    "GPC3", "SDC1", "SDC4", "CSPG4", "CSPG5", "PTPRZ1", "HAPLN1", "HAPLN2", "HAPLN3", "HAPLN4",
    "MMP1", "MMP2", "MMP3", "MMP7", "MMP9", "MMP11", "MMP12", "MMP13", "MMP14", "MMP16", "ADAMTS1", "ADAMTS4",
    "ADAMTS5", "ADAMTS8", "ADAM8", "HPSE", "PLAU", "PLAT",
    "TIMP1", "TIMP2", "TIMP3", "SERPINE1",
    "LOX", "LOXL1", "LOXL2", "LOXL3", "LOXL4", "TGM2", "P4HA1", "P4HA2", "P4HA3", "PLOD1", "PLOD2", "PLOD3",
    "BMP1", "PCOLCE",
    "HAS1", "HAS2", "HAS3", "HYAL1", "HYAL2",
]
CHECK_GENES = ["CTSD", "IGFBP2", "IGFBP7", "LGALS3"]
SUPP_4A = pd.read_csv(PATHS["supp_table_4a"])

pd.set_option("display.max_rows", 200)
PANELS = {"suppfig5c_ECM_nhood_TIL_zone_bubble_binary": "Supp Fig 5c", "fig5d_ecm_fractions_by_zone": "Fig 5d",
          "fig5e_collagen_engaged_vs_cold_combined": "Fig 5e", "suppfig5d_matrisome_total_per_zone": "Supp Fig 5d",
          "suppfig5e_matrisome_compositional_heatmap": "Supp Fig 5e"}   # also written to OUT_ROOT/panels/

ZONES_DF = zones_with_groups()   # spots; TIL_group is left out for sections without a TIL+ spot
GROUPS = til_groups(QUANTILE_CUTOFF)
CUTOFF = GROUPS.attrs["cutoff"]
ZONE_GROUPS = GROUPS["TIL_group_zones"]
print(f"TIL groups {GROUPS['TIL_group'].value_counts().to_dict()}, cutoff {CUTOFF:.4f}; zone comparisons "
      f"{ZONE_GROUPS.value_counts().to_dict()}, {int(ZONE_GROUPS.isna().sum())} sections without a TIL+ spot left out")

In [ ]:
def save(fig, stem, data):
    output.save(OUT, stem, fig, data, panel=PANELS.get(stem))


def make_bubble(df_in, value_col, value_label, cmap, vmin, vmax, title_extra, stem, top_n):
    df_res = df_in.copy()
    # FDR within zone
    df_res["fdr"] = np.nan
    for zone in ZONES_5C:
        mask = df_res["zone"] == zone
        if mask.sum() > 0:
            _, fdr, _, _ = multipletests(df_res.loc[mask, "pval"].fillna(1.0), method="fdr_bh")
            df_res.loc[mask, "fdr"] = fdr
    df_res["neglog10fdr"] = -np.log10(df_res["fdr"].clip(lower=1e-300))
    df_res["significant"] = df_res["fdr"] < FDR_THRESH

    print(f"\nBubble plot: {title_extra}")
    print(f"Total tests: {len(df_res)}")
    print(f"Significant (FDR<{FDR_THRESH}) per zone:")
    print(df_res.groupby("zone")["significant"].sum().to_string())

    # Significant in any zone, top N per direction, then the remaining neighborhoods for context
    sig_any = df_res[df_res["significant"]]["ecm_nhood"].unique()
    print(f"ECM neighborhoods significant in >= 1 zone: {len(sig_any)}")
    sig_df = df_res[(df_res["ecm_nhood"].isin(sig_any)) & (df_res["significant"])].copy()
    if sig_df.empty:
        print("No significant neighborhoods, showing all.")
        sort_order = sorted(df_res["ecm_nhood"].unique())
    else:
        best_per = sig_df.loc[sig_df.groupby("ecm_nhood")["fdr"].idxmin()]
        pos_all = best_per[best_per[value_col] > 0].sort_values(value_col, ascending=False)
        neg_all = best_per[best_per[value_col] < 0].sort_values(value_col, ascending=True)
        pos_cts = pos_all["ecm_nhood"].head(top_n).tolist()
        neg_cts = neg_all["ecm_nhood"].head(top_n).tolist()
        print(f"Positive: {len(pos_all)} total, showing {len(pos_cts)}")
        print(f"Negative: {len(neg_all)} total, showing {len(neg_cts)}")
        sort_ref = df_res[df_res["zone"] == SORT_ZONE].set_index("ecm_nhood")[value_col]
        pos_order = sorted(pos_cts, key=lambda ct: sort_ref.get(ct, 0), reverse=True)
        neg_order = sorted(neg_cts, key=lambda ct: sort_ref.get(ct, 0), reverse=True)
        sort_order = pos_order + neg_order
        sort_order = sort_order + [n for n in sorted(df_res["ecm_nhood"].unique()) if n not in sort_order]
    df_plot = df_res[df_res["ecm_nhood"].isin(sort_order)].copy()
    print(f"Final: {len(sort_order)} neighborhoods x {len(ZONES_5C)} zones")

    v_min, v_max = vmin, vmax
    if v_min is None or v_max is None:
        abs_max = df_plot[value_col].abs().quantile(0.95)
        v_min, v_max = -abs_max, abs_max

    with scanpy_style(200):
        zone_y = {z: i for i, z in enumerate(ZONES_5C)}
        ct_x = {ct: i for i, ct in enumerate(sort_order)}
        fig, ax = plt.subplots(figsize=figsize)
        ax.set_facecolor(bg_color)
        size_vals = df_plot["neglog10fdr"].values
        size_min = np.nanmin(size_vals)
        size_max = np.nanmax(size_vals)
        if size_max == size_min:
            size_max = size_min + 1

        def scale_size(v):
            frac = (v - size_min) / (size_max - size_min)
            return min_dot_size + frac * (max_dot_size - min_dot_size)

        for _, row in df_plot.iterrows():
            x = ct_x.get(row["ecm_nhood"])
            y = zone_y.get(row["zone"])
            if x is None or y is None:
                continue
            ec = sig_edgecolor if row["significant"] else nonsig_edgecolor
            ew = sig_edgewidth if row["significant"] else nonsig_edgewidth
            ax.scatter(x, y, s=scale_size(row["neglog10fdr"]), c=[row[value_col]], cmap=cmap, vmin=v_min, vmax=v_max,
                       edgecolors=ec, linewidths=ew, zorder=3)

        sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=v_min, vmax=v_max))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, shrink=0.8, aspect=20, pad=0.02)
        cbar.set_label(value_label, fontsize=axis_label_fontsize)
        cbar.ax.tick_params(labelsize=tick_fontsize)
        ax.set_xticks(range(len(sort_order)))
        ax.set_xticklabels(sort_order, rotation=nhood_rotation, fontsize=nhood_fontsize, ha="right",
                           rotation_mode="anchor")
        ax.set_yticks(range(len(ZONES_5C)))
        ax.set_yticklabels(ZONES_5C, fontsize=tick_fontsize)
        ax.set_xlim(-0.5, len(sort_order) - 0.5)
        ax.set_ylim(-0.5, len(ZONES_5C) - 0.5)
        for i in range(len(sort_order)):
            ax.axvline(i, color="0.94", linewidth=0.3, zorder=1)
        for i in range(len(ZONES_5C)):
            ax.axhline(i, color="0.94", linewidth=0.3, zorder=1)
        ax.set_xlabel("ECM neighborhood", fontsize=axis_label_fontsize)
        ax.set_ylabel("TIL zone", fontsize=axis_label_fontsize)
        ax.set_title(f"ECM neighborhoods across TIL zones — {title_extra}\n"
                     f"(FDR < {FDR_THRESH} in ≥1 zone, Top 8 NHoods)", fontsize=title_fontsize)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        legend_vals = [round(v, 1) for v in np.linspace(size_min, size_max, 4)]
        size_handles = [Line2D([0], [0], marker="o", linestyle="None", color="0.5", markersize=np.sqrt(scale_size(v)),
                               label=f"{v:.1f}", markeredgecolor="0.5", markeredgewidth=0.5) for v in legend_vals]
        sig_handle = Line2D([0], [0], marker="o", linestyle="None", color="0.7", markersize=8, markeredgecolor="black",
                            markeredgewidth=1.5, label=f"FDR < {FDR_THRESH}")
        ax.legend(handles=size_handles + [sig_handle], title="-log₁₀(FDR)", title_fontsize=legend_fontsize,
                  fontsize=legend_fontsize, loc="upper left", bbox_to_anchor=(1.12, 1.0), frameon=False,
                  handletextpad=1.5, labelspacing=1.8)
        plt.tight_layout()
        save(fig, stem, df_res)
        plt.show()
        plt.close(fig)

    pivot_val = df_plot.pivot(index="zone", columns="ecm_nhood", values=value_col)
    pivot_val = pivot_val[[c for c in sort_order if c in pivot_val.columns]].round(4)
    pivot_fdr = df_plot.pivot(index="zone", columns="ecm_nhood", values="fdr")
    pivot_fdr = pivot_fdr[[c for c in sort_order if c in pivot_fdr.columns]].round(4)
    print(f"\n{value_label}:")
    print(pivot_val.to_string())
    print("\nFDR:")
    print(pivot_fdr.to_string())
    return df_res, df_plot


def star(p):
    if p is None or np.isnan(p):
        return "ns"
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 5e-2 else "ns"


def paired_panel(ax, data, face_color, edge_color, title):
    n = len(data)
    if n == 0:
        ax.set_axis_off()
        return
    for _, row in data.iterrows():
        ax.plot([0, 1], [row["cold"], row["engaged"]], color=edge_color, alpha=0.20, linewidth=0.6, zorder=2)
    rng = np.random.RandomState(42)
    j_cold, j_engaged = rng.uniform(-0.06, 0.06, n), rng.uniform(-0.06, 0.06, n)
    ax.scatter(0 + j_cold, data["cold"], s=12, color=edge_color, alpha=0.55, zorder=3, edgecolors="none")
    ax.scatter(1 + j_engaged, data["engaged"], s=12, color=edge_color, alpha=0.55, zorder=3, edgecolors="none")
    bp = ax.boxplot([data["cold"].values, data["engaged"].values], positions=[0, 1], widths=0.45,
                    patch_artist=True, showfliers=False, zorder=4)
    for box in bp["boxes"]:
        box.set(facecolor=face_color, edgecolor=edge_color, linewidth=1.0, alpha=0.95)
    for w in bp["whiskers"]:
        w.set(color="#444444", linewidth=0.7)
    for c in bp["caps"]:
        c.set(color="#444444", linewidth=0.7)
    for m in bp["medians"]:
        m.set(color="black", linewidth=1.4)
    diff = (data["engaged"] - data["cold"]).values
    p = wilcoxon(diff).pvalue
    pos = (diff > 0).sum()
    ax.text(0.5, 0.97, f"n = {n} patients\np = {p:.2e}\n{pos}/{n} engaged > cold", transform=ax.transAxes,
            ha="center", va="top", fontsize=9, color="#222222",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="#fafafa", edgecolor="#cccccc", linewidth=0.5))
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["TIL−\n(immune-cold)", "TIL+ / BZ1 / BZ2\n(immune-engaged)"], fontsize=10)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylabel("Per-patient mean collagen panel\n(log-normalized)", fontsize=11)
    ax.grid(False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.text(0.02, 0.98, title, transform=ax.transAxes, ha="left", va="top", fontsize=12, fontweight="semibold",
            color=edge_color)


def per_sample_means(values_df, label):
    work = values_df.copy()
    work["sample"], work["TIL_zone"], work["TIL_group"] = (spot_df_v["sample"].values, spot_df_v["TIL_zone"].values,
                                                           spot_df_v["TIL_group"].values)
    counts = work.groupby(["sample", "TIL_zone"], observed=True).size().reset_index(name="n")
    work = work.merge(counts[counts["n"] >= MIN_SPOTS][["sample", "TIL_zone"]], on=["sample", "TIL_zone"])
    long = work.melt(id_vars=["sample", "TIL_zone", "TIL_group"], value_vars=CATEGORY_ORDER, var_name="category",
                     value_name=label)
    return long.groupby(["sample", "TIL_zone", "TIL_group", "category"], observed=True)[label].mean().reset_index()


def test_per_zone_category(df, value_col):
    records = []
    for zone in ZONES:
        for cat in CATEGORY_ORDER:
            sel = (df["TIL_zone"] == zone) & (df["category"] == cat)
            h = df[sel & (df["TIL_group"] == "TIL_high")][value_col].dropna().values
            l = df[sel & (df["TIL_group"] == "TIL_low")][value_col].dropna().values
            if len(h) < 3 or len(l) < 3:
                records.append({"zone": zone, "category": cat, "n_high": len(h), "n_low": len(l),
                                "median_high": np.nan, "median_low": np.nan, "p": np.nan})
                continue
            try:
                p = mannwhitneyu(h, l, alternative="two-sided").pvalue
            except ValueError:
                p = np.nan
            records.append({"zone": zone, "category": cat, "n_high": len(h), "n_low": len(l),
                            "median_high": float(np.median(h)), "median_low": float(np.median(l)),
                            "diff_median": float(np.median(h)) - float(np.median(l)), "p": p})
    res = pd.DataFrame(records)
    mask = res["p"].notna()
    res["fdr"] = np.nan
    if mask.any():
        res.loc[mask, "fdr"] = multipletests(res.loc[mask, "p"], method=FDR_METHOD)[1]
    return res


def box_pair(ax, vals, x, face, edge, width, dot_size=4, lw=0.9, wlw=0.6, mlw=1.1):
    bp = ax.boxplot(vals, positions=[x], widths=width, patch_artist=True, showfliers=False, zorder=2)
    for box in bp["boxes"]:
        box.set(facecolor=face, edgecolor=edge, linewidth=lw)
    for w in bp["whiskers"]:
        w.set(color="#444444", linewidth=wlw)
    for c in bp["caps"]:
        c.set(color="#444444", linewidth=wlw)
    for m in bp["medians"]:
        m.set(color="black", linewidth=mlw)
    jitter = (np.random.RandomState(42).rand(len(vals)) - 0.5) * (GROUP_W * 0.4)
    ax.scatter(x + jitter, vals, s=dot_size, color=edge, alpha=0.45 if dot_size == 4 else 0.4, zorder=3,
               edgecolors="none")


def stars(q):
    if pd.isna(q):
        return ""
    return "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else ""


def code_cells(path):
    # (label, code) per notebook code cell or per script, comments stripped
    def strip(text):
        return "\n".join(line.split("#")[0] for line in text.splitlines())
    if path.suffix == ".ipynb":
        cells = json.loads(path.read_text(errors="ignore"))["cells"]
        return [(f"{path.name}[{i}]", strip("".join(c["source"]))) for i, c in enumerate(cells)
                if c["cell_type"] == "code"]
    return [(path.name, strip(path.read_text(errors="ignore")))]

## Expression

Loads the Visium cohort with the standard loader exclusions and keeps the spot annotations and the matrisome genes used below.

In [ ]:
# Visium expression for the matrisome genes, log_norm_count layer

# Config
LAYER = "log_norm_count"

wanted = list(dict.fromkeys([g for v in CATEGORY_GENES.values() for g in v] + COLLAGEN_PANEL))
table_genes = list(dict.fromkeys(wanted + SUMMARY_LIST_119 + CHECK_GENES + SUPP_4A["target"].tolist()))
adata = AllDataCenters.load_spatial_adata(genes=table_genes, list_sample_id=visium.sections(), normalization=LAYER)
VAR_NAMES = set(adata.var_names.astype(str))
present = [g for g in wanted if g in VAR_NAMES]
x = adata[:, present].layers[LAYER]
EXPR = pd.DataFrame(np.asarray(x, dtype=np.float64), index=adata.obs_names.astype(str), columns=present)
obs = pd.DataFrame({"sample": adata.obs["sample"].astype(str).values}, index=adata.obs_names.astype(str))
del adata, x
gc.collect()
obs["TIL_zone"] = ZONES_DF["TIL_zone"].reindex(obs.index).astype(str)
obs["TIL_group"] = ZONES_DF["TIL_group"].reindex(obs.index)
print(f"{len(EXPR):,} spots; {len(present)} of {len(wanted)} panel genes present; absent "
      f"{sorted(set(wanted) - VAR_NAMES) or 'none'}")
print(f"Spots missing from the TIL zone file: {int(obs['TIL_zone'].isin(['nan', 'None']).sum())}")
print(f"TIL_group values: {obs['TIL_group'].value_counts(dropna=False).to_dict()}")

## Supp Fig 5c: ECM neighborhoods by TIL zone

Within each TIL zone, the fraction of each section's spots in each ECM neighborhood (ECM_5 and ECM_6 left out) is compared between TIL-high and TIL-low sections with Welch's t-test. P values are Benjamini-Hochberg corrected within each zone, and the plot keeps neighborhoods significant in at least one zone and appends the rest.

In [ ]:
# Supp Fig 5c, ECM neighborhood differences between TIL-high and TIL-low sections across TIL zones

# Config
ZONES_5C = ["TIL-", "BZ2", "BZ1", "TIL+"]
TEST = "ttest"                 # "mw", "ttest", or "both" (MW primary)
FDR_THRESH = 0.05
TOP_N = 15
SORT_ZONE = "TIL+"
EXCLUDE_NHOODS = ["ECM_5", "ECM_6"]
figsize = (13, 4)
min_dot_size = 20
max_dot_size = 400
sig_edgecolor = "black"
sig_edgewidth = 1.5
nonsig_edgecolor = "0.7"
nonsig_edgewidth = 0.5
bg_color = "white"
title_fontsize = 12
axis_label_fontsize = 10
tick_fontsize = 9
nhood_fontsize = 9
nhood_rotation = 45
legend_fontsize = 9
cmap_binary = "RdBu_r"
vmin_binary = None
vmax_binary = None

obs_5c = ZONES_DF[["sample", "TIL_zone", "TIL_group"]].copy()
obs_5c["Sample_TIL_Group"] = obs_5c["TIL_group"].map({"TIL_high": "TIL-High", "TIL_low": "TIL-Low"})

ecm_5c = pd.read_csv(PATHS["ecm"])
required_cols = {"spot_id", "sample", ECM_KEY}
if not required_cols.issubset(ecm_5c.columns):
    raise KeyError(f"ECM file must contain {required_cols}. Found {set(ecm_5c.columns)}")
obs_for_merge = obs_5c.reset_index()[["spot_id", "sample", "TIL_zone", "Sample_TIL_Group"]]
ecm_merged = ecm_5c.merge(obs_for_merge, on=["spot_id", "sample"], how="inner")
print(f"ECM spots merged with obs: {len(ecm_merged)}")

if TEST == "both":
    test_label = "Mann-Whitney (primary) + Welch t-test"
elif TEST == "mw":
    test_label = "Mann-Whitney U"
else:
    test_label = "Welch's t-test"

# Per-zone ECM neighborhood fractions per section, then the group test
results_binary = []
for zone in ZONES_5C:
    ecm_zone = ecm_merged[ecm_merged["TIL_zone"] == zone].copy()
    if ecm_zone.empty:
        print(f"WARNING: no ECM spots in zone {zone}")
        continue
    ecm_counts = ecm_zone.groupby(["sample", ECM_KEY], observed=True).size().reset_index(name="count")
    ecm_pivot = ecm_counts.pivot_table(index="sample", columns=ECM_KEY, values="count", fill_value=0)
    ecm_frac = ecm_pivot.div(ecm_pivot.sum(axis=1), axis=0)
    ecm_frac.columns = [f"ECM_{c}" for c in ecm_frac.columns]
    ecm_frac = ecm_frac[[c for c in ecm_frac.columns if c not in EXCLUDE_NHOODS]]
    ecm_features = ecm_frac.columns.tolist()
    if not ecm_features:
        continue

    meta_group = ecm_zone.drop_duplicates("sample").set_index("sample")["Sample_TIL_Group"]
    ecm_frac["Sample_TIL_Group"] = meta_group.reindex(ecm_frac.index)
    ecm_frac_clean = ecm_frac.dropna(subset=["Sample_TIL_Group"]).copy()
    g_high = ecm_frac_clean[ecm_frac_clean["Sample_TIL_Group"] == "TIL-High"]
    g_low = ecm_frac_clean[ecm_frac_clean["Sample_TIL_Group"] == "TIL-Low"]

    if len(g_high) >= 3 and len(g_low) >= 3:
        for col in ecm_features:
            a = g_high[col].dropna().to_numpy()
            b = g_low[col].dropna().to_numpy()
            if a.size < 3 or b.size < 3:
                continue
            mh, ml = float(a.mean()), float(b.mean())
            if TEST == "both":
                _, p_mw = mannwhitneyu(a, b, alternative="two-sided")
                _, p_tt = ttest_ind(a, b, equal_var=False)
                p = p_mw
            elif TEST == "mw":
                _, p_mw = mannwhitneyu(a, b, alternative="two-sided")
                p, p_tt = p_mw, np.nan
            else:
                _, p_tt = ttest_ind(a, b, equal_var=False)
                p, p_mw = p_tt, np.nan
            results_binary.append({"zone": zone, "ecm_nhood": col, "mean_high": mh, "mean_low": ml,
                                   "diff": mh - ml, "pval": p, "pval_mw": p_mw, "pval_ttest": p_tt,
                                   "n_high": len(a), "n_low": len(b)})

df_binary = pd.DataFrame(results_binary)
if not df_binary.empty:
    df_binary_res, df_binary_plot = make_bubble(
        df_binary, value_col="diff", value_label="Mean difference (TIL-High − TIL-Low)", cmap=cmap_binary,
        vmin=vmin_binary, vmax=vmax_binary, title_extra=f"TIL-High vs TIL-Low ({test_label})",
        stem="suppfig5c_ECM_nhood_TIL_zone_bubble_binary", top_n=TOP_N)
    if TEST == "both":
        df_binary_res["fdr_ttest"] = np.nan
        for zone in ZONES_5C:
            mask = df_binary_res["zone"] == zone
            if mask.sum() > 0:
                df_binary_res.loc[mask, "fdr_ttest"] = multipletests(
                    df_binary_res.loc[mask, "pval_ttest"].fillna(1.0), method="fdr_bh")[1]
        both_sig = (df_binary_res["fdr"] < FDR_THRESH) & (df_binary_res["fdr_ttest"] < FDR_THRESH)
        mw_only = (df_binary_res["fdr"] < FDR_THRESH) & (df_binary_res["fdr_ttest"] >= FDR_THRESH)
        tt_only = (df_binary_res["fdr"] >= FDR_THRESH) & (df_binary_res["fdr_ttest"] < FDR_THRESH)
        print(f"\nConcordance: both={both_sig.sum()}, MW only={mw_only.sum()}, t-test only={tt_only.sum()}")
    df_binary_res.to_csv(OUT / "suppfig5c_ECM_nhood_TIL_zone_binary_bubble.csv", index=False)
else:
    print("No binary results to plot.")

In [ ]:
# Supp Fig 5c compared with the published table
pub = pd.read_csv(PUBLISHED["suppfig5c"])
m = df_binary_res.merge(pub, on=["zone", "ecm_nhood"], suffixes=("", "_pub"))
print(f"Rows: this run {len(df_binary_res)}, published {len(pub)}, matched {len(m)}")
for zone, gz in m.groupby("zone"):
    print(f"  {zone}: n_low {sorted(gz['n_low'].unique())} (published {sorted(gz['n_low_pub'].unique())}), "
          f"max |fdr diff| {np.abs(gz['fdr'] - gz['fdr_pub']).max():.2g}, "
          f"significance differs for {gz.loc[gz['significant'] != gz['significant_pub'], 'ecm_nhood'].tolist()}")

## Fig 5d: ECM neighborhood fractions by zone

Per-sample fraction of each zone's spots in five ECM neighborhoods, TIL-high against TIL-low, on a log axis. Zero fractions are drawn at a floor of 10⁻³ and stars mark two-sided Mann-Whitney p values.

In [ ]:
# Fig 5d, per-patient ECM neighborhood fraction within each TIL zone, split by TIL group (log y)

# Config
ECMS = [0, 1, 2, 7, 8]
MIN_ZONE_SPOTS = 0          # drop patient-zones with fewer spots than this (0 keeps all)
FLOOR = 1e-3                # zero fractions are floored here for the log axis
COL_HIGH_EDGE, COL_HIGH_FILL = "#C0392B", "#F2C0BC"
COL_LOW_EDGE, COL_LOW_FILL = "#2166AC", "#C6DBEF"

ecm = pd.read_csv(PATHS["ecm"], usecols=["spot_id", "sample", ECM_KEY]).dropna(subset=[ECM_KEY])
ecm[ECM_KEY] = ecm[ECM_KEY].astype(int)
ecm["TIL_zone"] = ecm["spot_id"].astype(str).map(ZONES_DF["TIL_zone"])
ecm = ecm[ecm["TIL_zone"].isin(ZONES)].copy()
ecm["patient"] = ecm["sample"].str.replace(r"_vis$", "", regex=True)
# Section to TIL group, keyed like TIL_groups.csv (sample without '_vis'), without TIL-negative sections
grp_map = pd.Series(ZONE_GROUPS.values, index=ZONE_GROUPS.index.str.replace(r"_vis$", "", regex=True))
ecm["TIL_group"] = ecm["patient"].map(grp_map)
ecm = ecm.dropna(subset=["TIL_group"])

tot = ecm.groupby(["patient", "TIL_zone"]).size().rename("tot").reset_index()
if MIN_ZONE_SPOTS > 0:
    tot = tot[tot["tot"] >= MIN_ZONE_SPOTS]
num = (ecm[ecm[ECM_KEY].isin(ECMS)].groupby(["patient", "TIL_zone", ECM_KEY]).size().rename("n").reset_index())
grid = tot.merge(pd.DataFrame({ECM_KEY: ECMS}), how="cross")
grid = grid.merge(num, on=["patient", "TIL_zone", ECM_KEY], how="left")
grid["n"] = grid["n"].fillna(0)
grid["frac"] = grid["n"] / grid["tot"]
grid["TIL_group"] = grid["patient"].map(grp_map)
grid = grid.dropna(subset=["TIL_group"])
grid["frac_disp"] = grid["frac"].clip(lower=FLOOR)
print(f"Patients: {grid['patient'].nunique()}  |  by group: "
      f"{grid.drop_duplicates('patient')['TIL_group'].value_counts().to_dict()}")
print(f"Zones x ECM cells plotted: {len(ZONES) * len(ECMS)}  |  MIN_ZONE_SPOTS={MIN_ZONE_SPOTS}")

records = []
rng = np.random.default_rng(0)
fig = plt.figure(figsize=(15, 5.2))
gs = GridSpec(1, len(ZONES), wspace=0.06)
axes = []
ymax_global = grid["frac_disp"].max()
for zi, z in enumerate(ZONES):
    ax = fig.add_subplot(gs[0, zi], sharey=axes[0] if axes else None)
    axes.append(ax)
    ax.set_yscale("log")
    for ei, e in enumerate(ECMS):
        cell = grid[(grid["TIL_zone"] == z) & (grid[ECM_KEY] == e)]
        for grp_name, off, fe, ff in [("TIL_high", -0.2, COL_HIGH_EDGE, COL_HIGH_FILL),
                                      ("TIL_low", 0.2, COL_LOW_EDGE, COL_LOW_FILL)]:
            d = cell[cell["TIL_group"] == grp_name]["frac_disp"].values
            if len(d) == 0:
                continue
            xp = ei + off
            ax.boxplot([d], positions=[xp], widths=0.34, patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor=ff, edgecolor=fe, linewidth=1.0),
                       whiskerprops=dict(color=fe, linewidth=1.0), capprops=dict(color=fe, linewidth=1.0),
                       medianprops=dict(color="black", linewidth=1.3))
            ax.scatter(xp + rng.uniform(-0.07, 0.07, len(d)), d, s=7, color=fe, alpha=0.4, linewidth=0, zorder=3)
        hi = cell[cell["TIL_group"] == "TIL_high"]["frac"].values
        lo = cell[cell["TIL_group"] == "TIL_low"]["frac"].values
        try:
            pval = mannwhitneyu(hi, lo, alternative="two-sided").pvalue
        except ValueError:
            pval = np.nan
        records.append({"zone": z, "ECM": f"ECM_{e}", "n_high": len(hi), "n_low": len(lo),
                        "median_high": np.median(hi) if len(hi) else np.nan,
                        "median_low": np.median(lo) if len(lo) else np.nan, "p": pval, "star": star(pval)})
        ytop = max(cell["frac_disp"].max(), FLOOR)
        ax.text(ei, ytop * 1.5, star(pval), ha="center", va="bottom", fontsize=11)
    ax.set_xticks(range(len(ECMS)))
    ax.set_xticklabels([f"ECM_{e}" for e in ECMS], rotation=45, ha="right", fontsize=9)
    ax.set_xlim(-0.6, len(ECMS) - 0.4)
    ax.set_title(z, fontsize=13, fontweight="bold")
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    if zi == 0:
        ax.set_ylabel("Per-patient ECM neighborhood fraction", fontsize=11)
    else:
        ax.tick_params(labelleft=False, left=False)
        ax.spines["left"].set_visible(False)
axes[0].set_ylim(FLOOR / 1.6, ymax_global * 4)
axes[-1].legend(handles=[Patch(facecolor=COL_HIGH_FILL, edgecolor=COL_HIGH_EDGE, label="TIL-high"),
                         Patch(facecolor=COL_LOW_FILL, edgecolor=COL_LOW_EDGE, label="TIL-low")],
                loc="upper right", fontsize=9, framealpha=0.9)
save(fig, "fig5d_ecm_fractions_by_zone", grid)
plt.show()
plt.close(fig)
fig5d = pd.DataFrame(records)
fig5d["median diff"] = fig5d["median_high"] - fig5d["median_low"]
display(fig5d.round(4).set_index(["zone", "ECM"]))
fig5d.to_csv(OUT / "fig5d_tests.csv", index=False)

In [ ]:
# Fig 5d: the same tests on section-zones with >= 20 spots, the filter of the Final_Notebooks tables,
# and the per-section fractions compared with the published ones

# Config
MIN_ZONE_SPOTS_FINAL = 20

grid20 = grid[grid["tot"] >= MIN_ZONE_SPOTS_FINAL]
rec20 = []
for (z, e), cell in grid20.groupby(["TIL_zone", ECM_KEY]):
    hi = cell[cell["TIL_group"] == "TIL_high"]["frac"].values
    lo = cell[cell["TIL_group"] == "TIL_low"]["frac"].values
    pval = mannwhitneyu(hi, lo, alternative="two-sided").pvalue
    rec20.append({"zone": z, "ECM": f"ECM_{e}", "n_high": len(hi), "n_low": len(lo), "median_high": np.median(hi),
                  "median_low": np.median(lo), "p": pval, "star": star(pval)})
fig5d_20 = pd.DataFrame(rec20).set_index(["zone", "ECM"]).reindex(fig5d.set_index(["zone", "ECM"]).index)
pub_t = pd.read_csv(PUBLISHED["fig5d_tests"]).rename(columns={"ecm": "ECM"}).set_index(["zone", "ECM"])
fig5d_20["star (MIN_ZONE_SPOTS=0, above)"] = fig5d.set_index(["zone", "ECM"])["star"]
fig5d_20["p published table"] = pub_t["p_mw"]
fig5d_20["star published table"] = pub_t["sig"]
print(f"Fig 5d tests with section-zones of >= {MIN_ZONE_SPOTS_FINAL} spots:")
display(fig5d_20.round(4))
fig5d_20.to_csv(OUT / "fig5d_tests_min20spots.csv")

pub = pd.read_csv(PUBLISHED["fig5d"])
pub["patient"] = pub["sample"].str.replace(r"_vis$", "", regex=True)
b = grid.merge(pub[pub[ECM_KEY].isin(ECMS)], on=["patient", "TIL_zone", ECM_KEY], suffixes=("", "_pub"))
print(f"Per-section fractions in both: {len(b)}; max |fraction - published| = {np.abs(b['frac'] - b['fraction']).max():.3g}; "
      f"TIL group differs in {int((b['TIL_group'] != b['TIL_group_pub']).sum())} rows")

## Fig 5e: collagen expression in immune-engaged and immune-cold spots

Within each sample, mean expression of the 10-gene collagen panel over immune-engaged spots (TIL+, BZ1 and BZ2) and over immune-cold spots (TIL−), each with at least 20 spots, compared by paired Wilcoxon signed-rank test. The first figure splits TIL-high and TIL-low samples and the second, Fig 5e, combines them.

In [ ]:
# Fig 5e, per-spot collagen expression in immune-engaged versus immune-cold zones

# Config
ENGAGED_ZONES = ["TIL+", "BZ1", "BZ2"]
COLD_ZONES = ["TIL-"]
MIN_SPOTS = 20
COL_HIGH_FACE, COL_HIGH_EDGE = "#f4c8c1", "#b2182b"
COL_LOW_FACE, COL_LOW_EDGE = "#c5d9ec", "#2166ac"
COL_COMBO_FACE, COL_COMBO_EDGE = "#dddddd", "#444444"

available = [g for g in COLLAGEN_PANEL if g in EXPR.columns]
if set(COLLAGEN_PANEL) - set(available):
    print(f"Collagen genes not in the expression object: {sorted(set(COLLAGEN_PANEL) - set(available))}")
panel_mean = EXPR[available].mean(axis=1)
print(f"Panel: {len(available)} collagens averaged per spot")
work = obs.assign(collagen=panel_mean.values)
work["compartment"] = work["TIL_zone"].map(
    lambda z: "engaged" if z in ENGAGED_ZONES else ("cold" if z in COLD_ZONES else None))
work = work[work["compartment"].notna() & work["TIL_group"].isin(["TIL_high", "TIL_low"])]
per_patient = (work.groupby(["sample", "TIL_group", "compartment"], observed=True)
               .agg(mean_collagen=("collagen", "mean"), n_spots=("collagen", "size")).reset_index())
per_patient = per_patient[per_patient["n_spots"] >= MIN_SPOTS]
wide = (per_patient.pivot_table(index=["sample", "TIL_group"], columns="compartment", values="mean_collagen")
        .reset_index().dropna(subset=["engaged", "cold"]))

print(f"Paired Wilcoxon: engaged ({'+'.join(ENGAGED_ZONES)}) vs cold ({COLD_ZONES[0]}), "
      f"min {MIN_SPOTS} spots per sample per compartment")
stats_records = []
for grp_label, grp_filter in [("TIL-high", lambda d: d["TIL_group"] == "TIL_high"),
                              ("TIL-low", lambda d: d["TIL_group"] == "TIL_low"),
                              ("Combined", lambda d: d["TIL_group"].isin(["TIL_high", "TIL_low"]))]:
    g = wide[grp_filter(wide)]
    n = len(g)
    if n < 5:
        print(f"{grp_label}: too few patients ({n})")
        continue
    diff = (g["engaged"] - g["cold"]).values
    p = wilcoxon(diff).pvalue
    pos = int((diff > 0).sum())
    stats_records.append({"group": grp_label, "n": n, "engaged_median": g["engaged"].median(),
                          "cold_median": g["cold"].median(), "delta_median": np.median(diff), "p_wilcoxon": p,
                          "engaged > cold": f"{pos}/{n}", "pct_directional": pos / n * 100})
stats_df = pd.DataFrame(stats_records)
display(stats_df.set_index("group").round(4))
stats_df.to_csv(OUT / "fig5e_collagen_engaged_vs_cold.csv", index=False)

with scanpy_style(150):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
    paired_panel(axes[0], wide[wide["TIL_group"] == "TIL_high"], COL_HIGH_FACE, COL_HIGH_EDGE, "TIL-high patients")
    paired_panel(axes[1], wide[wide["TIL_group"] == "TIL_low"], COL_LOW_FACE, COL_LOW_EDGE, "TIL-low patients")
    plt.tight_layout()
    save(fig, "fig5e_collagen_split_by_group", wide)
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(5, 5))
    data_all = wide.reset_index(drop=True)
    n = len(data_all)
    for _, row in data_all.iterrows():
        color = COL_HIGH_EDGE if row["TIL_group"] == "TIL_high" else COL_LOW_EDGE
        ax.plot([0, 1], [row["cold"], row["engaged"]], color=color, alpha=0.18, linewidth=0.6, zorder=2)
    rng = np.random.RandomState(42)
    j_cold, j_engaged = rng.uniform(-0.06, 0.06, n), rng.uniform(-0.06, 0.06, n)
    for i, row in data_all.iterrows():
        color = COL_HIGH_EDGE if row["TIL_group"] == "TIL_high" else COL_LOW_EDGE
        ax.scatter(0 + j_cold[i], row["cold"], s=14, color=color, alpha=0.55, zorder=3, edgecolors="none")
        ax.scatter(1 + j_engaged[i], row["engaged"], s=14, color=color, alpha=0.55, zorder=3, edgecolors="none")
    bp = ax.boxplot([data_all["cold"].values, data_all["engaged"].values], positions=[0, 1], widths=0.45,
                    patch_artist=True, showfliers=False, zorder=4)
    for box in bp["boxes"]:
        box.set(facecolor=COL_COMBO_FACE, edgecolor=COL_COMBO_EDGE, linewidth=1.0, alpha=0.85)
    for w in bp["whiskers"]:
        w.set(color="#444444", linewidth=0.7)
    for c in bp["caps"]:
        c.set(color="#444444", linewidth=0.7)
    for m in bp["medians"]:
        m.set(color="black", linewidth=1.4)
    diff = (data_all["engaged"] - data_all["cold"]).values
    p = wilcoxon(diff).pvalue
    pos = (diff > 0).sum()
    ax.text(0.5, 0.97, f"n = {n} patients\np = {p:.2e}\n{pos}/{n} engaged > cold", transform=ax.transAxes,
            ha="center", va="top", fontsize=9, color="#222222",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="#fafafa", edgecolor="#cccccc", linewidth=0.5))
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["TIL−\n(immune-cold)", "TIL+ / BZ1 / BZ2\n(immune-engaged)"], fontsize=10)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylabel("Per-patient mean collagen panel\n(log-normalized)", fontsize=11)
    ax.grid(False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.legend(handles=[Patch(facecolor=COL_HIGH_FACE, edgecolor=COL_HIGH_EDGE, label="TIL-high"),
                       Patch(facecolor=COL_LOW_FACE, edgecolor=COL_LOW_EDGE, label="TIL-low")],
              loc="lower right", frameon=False, fontsize=9)
    plt.tight_layout()
    save(fig, "fig5e_collagen_engaged_vs_cold_combined", data_all)
    plt.show()
    plt.close(fig)

In [ ]:
print("Published (Final_Notebooks/collagen_engaged_vs_cold_stats.csv):")
display(pd.read_csv(PUBLISHED["fig5e"]).set_index("group"))

## Supp Fig 5d and 5e: matrisome categories by zone

For each matrisome category, the absolute expression (sum of category genes per spot) and the compositional fraction (category sum over total matrisome sum per spot) are averaged per sample and zone (at least 20 spots), then compared between TIL-high and TIL-low with Mann-Whitney U tests and Benjamini-Hochberg correction across all zone and category tests of each analysis.

In [ ]:
# Supp Fig 5d and 5e, matrisome category analysis per TIL zone (absolute and compositional)

# Config
MIN_SPOTS = 20
FDR_METHOD = "fdr_bh"
COL_HIGH_FACE, COL_HIGH_EDGE = "#f4c8c1", "#b2182b"
COL_LOW_FACE, COL_LOW_EDGE = "#c5d9ec", "#2166ac"
GROUP_W = 0.40

gene_to_cat = {g: cat for cat, gs in CATEGORY_GENES.items() for g in gs if g in EXPR.columns}
print(f"ECM genes available: {len(gene_to_cat)} of {sum(len(v) for v in CATEGORY_GENES.values())} in panel")
for cat in CATEGORY_ORDER:
    print(f"  {cat}: {sum(1 for g in CATEGORY_GENES[cat] if g in gene_to_cat)}/{len(CATEGORY_GENES[cat])}")
cat_sums = pd.DataFrame(index=EXPR.index, columns=CATEGORY_ORDER, dtype=float)
for cat in CATEGORY_ORDER:
    genes_in_cat = [g for g in CATEGORY_GENES[cat] if g in gene_to_cat]
    cat_sums[cat] = EXPR[genes_in_cat].sum(axis=1) if genes_in_cat else 0.0
total_matrisome = cat_sums.sum(axis=1).replace(0, np.nan)
cat_fracs = cat_sums.div(total_matrisome, axis=0).fillna(0.0)
spot_df = obs[["sample", "TIL_zone", "TIL_group"]]
valid = spot_df["TIL_zone"].isin(ZONES) & spot_df["TIL_group"].isin(["TIL_high", "TIL_low"]) & total_matrisome.notna()
print(f"Valid spots: {int(valid.sum()):,} of {len(spot_df):,}")
cat_sums_v, cat_fracs_v, spot_df_v, total_v = cat_sums[valid], cat_fracs[valid], spot_df[valid], total_matrisome[valid]

abs_per_sample = per_sample_means(cat_sums_v, "value")
frac_per_sample = per_sample_means(cat_fracs_v, "value")
total_work = pd.DataFrame({"sample": spot_df_v["sample"].values, "TIL_zone": spot_df_v["TIL_zone"].values,
                           "TIL_group": spot_df_v["TIL_group"].values, "total": total_v.values})
total_counts = total_work.groupby(["sample", "TIL_zone"], observed=True).size().reset_index(name="n")
total_work = total_work.merge(total_counts[total_counts["n"] >= MIN_SPOTS][["sample", "TIL_zone"]],
                              on=["sample", "TIL_zone"])
total_per_sample = total_work.groupby(["sample", "TIL_zone", "TIL_group"], observed=True)["total"].mean().reset_index()

abs_stats = test_per_zone_category(abs_per_sample, "value")
frac_stats = test_per_zone_category(frac_per_sample, "value")
for stats, label in [(abs_stats, "ABSOLUTE matrisome expression (TIL-high vs TIL-low, per zone x category)"),
                     (frac_stats, "COMPOSITIONAL matrisome fraction (TIL-high vs TIL-low, per zone x category)")]:
    print(label)
    fmt = stats.copy()
    for c in ["median_high", "median_low", "diff_median"]:
        fmt[c] = fmt[c].round(4)
    for c in ["p", "fdr"]:
        fmt[c] = fmt[c].apply(lambda v: f"{v:.2e}" if pd.notna(v) else "n/a")
    display(fmt.set_index(["zone", "category"]))
abs_stats.to_csv(OUT / "matrisome_absolute_per_zone_category.csv", index=False)
frac_stats.to_csv(OUT / "matrisome_compositional_per_zone_category.csv", index=False)
abs_per_sample.to_csv(OUT / "matrisome_absolute_per_sample.csv", index=False)
frac_per_sample.to_csv(OUT / "matrisome_compositional_per_sample.csv", index=False)
total_per_sample.to_csv(OUT / "matrisome_total_per_sample.csv", index=False)
for stats, k in [(abs_stats, "abs"), (frac_stats, "frac")]:
    pub = pd.read_csv(PUBLISHED[k])
    m = stats.merge(pub, on=["zone", "category"], suffixes=("", "_pub"))
    for zone, gz in m.groupby("zone", sort=False):
        print(f"{k} vs published, {zone}: n equal {bool((gz['n_high'] == gz['n_high_pub']).all() and (gz['n_low'] == gz['n_low_pub']).all())}; "
              f"max |median diff| {np.nanmax(np.abs(gz['diff_median'] - gz['diff_median_pub'])):.2g}; "
              f"max |fdr diff| {np.nanmax(np.abs(gz['fdr'] - gz['fdr_pub'])):.2g}")

legend_handles = [Patch(facecolor=COL_HIGH_FACE, edgecolor=COL_HIGH_EDGE, label="TIL-high"),
                  Patch(facecolor=COL_LOW_FACE, edgecolor=COL_LOW_EDGE, label="TIL-low")]

with scanpy_style(150):
    # Plot 1, total matrisome expression per zone
    fig, ax = plt.subplots(figsize=(7, 5))
    for zi, zone in enumerate(ZONES):
        for grp, off, face, edge in [("TIL_high", -GROUP_W / 2, COL_HIGH_FACE, COL_HIGH_EDGE),
                                     ("TIL_low", GROUP_W / 2, COL_LOW_FACE, COL_LOW_EDGE)]:
            vals = total_per_sample[(total_per_sample["TIL_zone"] == zone)
                                    & (total_per_sample["TIL_group"] == grp)]["total"].dropna().values
            if len(vals) >= 3:
                box_pair(ax, vals, zi + off, face, edge, GROUP_W * 0.85)
    ax.set_xticks(range(len(ZONES)))
    ax.set_xticklabels(ZONES, fontsize=11)
    ax.set_ylabel("Per-patient mean total matrisome\nexpression (log-normalized sum)", fontsize=11)
    ax.grid(False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.legend(handles=legend_handles, loc="upper right", frameon=False, fontsize=10)
    plt.tight_layout()
    save(fig, "suppfig5d_matrisome_total_per_zone", total_per_sample)
    plt.show()
    plt.close(fig)

    # Plot 2, compositional difference heatmap with FDR stars
    heatmap = frac_stats.pivot_table(index="category", columns="zone", values="diff_median").reindex(
        index=CATEGORY_ORDER, columns=ZONES)
    fdr_mat = frac_stats.pivot_table(index="category", columns="zone", values="fdr").reindex(
        index=CATEGORY_ORDER, columns=ZONES)
    vmax = np.nanmax(np.abs(heatmap.values))
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(heatmap.values, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="none")
    ax.set_xticks(range(len(ZONES)))
    ax.set_xticklabels(ZONES, fontsize=11)
    ax.set_yticks(range(len(CATEGORY_ORDER)))
    ax.set_yticklabels(CATEGORY_ORDER, fontsize=11)
    ax.set_xlabel("TIL zone", fontsize=11, labelpad=8)
    ax.tick_params(length=0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    for i, cat in enumerate(CATEGORY_ORDER):
        for j, zone in enumerate(ZONES):
            v, q = heatmap.loc[cat, zone], fdr_mat.loc[cat, zone]
            if pd.notna(v):
                st = stars(q)
                ax.text(j, i, f"{v:+.3f}\n{st}" if st else f"{v:+.3f}", ha="center", va="center", fontsize=9,
                        color="white" if abs(v) > vmax * 0.55 else "#222222")
    cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02, shrink=0.8)
    cbar.set_label("Compositional fraction difference\n(TIL-high − TIL-low)", fontsize=10)
    cbar.ax.tick_params(labelsize=9)
    plt.tight_layout()
    save(fig, "suppfig5e_matrisome_compositional_heatmap", frac_stats)
    plt.show()
    plt.close(fig)

    # Plot 3, compositional fraction across zones, one panel per category
    n_cols = 4
    n_rows = int(np.ceil(len(CATEGORY_ORDER) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.5 * n_rows))
    axes = axes.flatten()
    for ai, cat in enumerate(CATEGORY_ORDER):
        ax = axes[ai]
        for zi, zone in enumerate(ZONES):
            for grp, off, face, edge in [("TIL_high", -GROUP_W / 2, COL_HIGH_FACE, COL_HIGH_EDGE),
                                         ("TIL_low", GROUP_W / 2, COL_LOW_FACE, COL_LOW_EDGE)]:
                vals = frac_per_sample[(frac_per_sample["TIL_zone"] == zone) & (frac_per_sample["TIL_group"] == grp)
                                       & (frac_per_sample["category"] == cat)]["value"].dropna().values
                if len(vals) >= 3:
                    box_pair(ax, vals, zi + off, face, edge, GROUP_W * 0.85, dot_size=3, lw=0.8, wlw=0.5, mlw=1.0)
        ax.set_xticks(range(len(ZONES)))
        ax.set_xticklabels(ZONES, fontsize=9)
        ax.set_ylabel("Fraction", fontsize=9)
        ax.set_title(cat, fontsize=10, loc="left")
        ax.grid(False)
        for s in ["top", "right"]:
            ax.spines[s].set_visible(False)
        ax.tick_params(labelsize=8)
    for ai in range(len(CATEGORY_ORDER), len(axes)):
        axes[ai].set_visible(False)
    axes[0].legend(handles=legend_handles, loc="upper right", frameon=False, fontsize=8)
    plt.tight_layout()
    save(fig, "matrisome_compositional_per_category", frac_per_sample)
    plt.show()
    plt.close(fig)

## Matrisome gene table

One row per gene in any matrisome list used in the analysis notebooks, with the analyses it entered. ECM neighborhood clustering and the matrisome category analysis share one 117-gene panel. GBM_MS_Summary holds a 119-gene list that differs from it by the genes flagged in the last column.

In [ ]:
# Matrisome gene table for the supplement


panel = {g: cat for cat, gs in CATEGORY_GENES.items() for g in gs}
genes = list(dict.fromkeys(list(panel) + SUMMARY_LIST_119))
table = pd.DataFrame({
    "gene": genes,
    "matrisome category": [panel.get(g, "") for g in genes],
    "ECM neighborhood clustering": [g in panel for g in genes],
    "matrisome category analysis (Supp Fig 5d, 5e)": [g in panel for g in genes],
    "collagen panel (Fig 5e)": [g in COLLAGEN_PANEL for g in genes],
    "in Visium gene space": [g in VAR_NAMES for g in genes],
    "in GBM_MS_Summary 119-gene list": [g in SUMMARY_LIST_119 for g in genes],
})
print(f"Panel genes: {len(panel)}; 119-gene list: {len(set(SUMMARY_LIST_119))}; union: {len(genes)}")
print(f"Only in the panel: {sorted(set(panel) - set(SUMMARY_LIST_119))}")
print(f"Only in the 119-gene list: {sorted(set(SUMMARY_LIST_119) - set(panel))}")
in_lists = {g: ("in a list" if g in genes else "in no list") for g in CHECK_GENES}
print(f"Genes to confirm against the supplementary table: {in_lists}")
display(table.set_index("gene"))
table.to_csv(OUT / "matrisome_gene_table.csv", index=False)

## Supp Table 4a: genes and the analyses that use them

Supp Table 4a (EFS `supplementary_tables/4a_ECM_genes.csv`, 123 genes) also lists CTSD, IGFBP2, IGFBP7
and LGALS3, which are in none of the lists above. For each of its genes, the code of the analysis files
(`analysis/` without `analysis/scratch/`, the `gbmcollaboration` package; comments stripped, so a gene
commented out of a list does not count) is searched for the gene symbol as a quoted string, and the
files (notebook cells) that use it are listed. This notebook is left out: its lists are the ones above
and `CHECK_GENES` only names the genes to check.

In [ ]:
# Config
SEARCH_DIRS = [config.REPO_ROOT / "analysis", config.REPO_ROOT / "gbmcollaboration"]
SKIP = {"fig5de_suppfig5cde_ECM_supplement.ipynb"}

code = [cc for d in SEARCH_DIRS for p in sorted(d.rglob("*")) if p.suffix in {".py", ".ipynb", ".R"}
        and p.name not in SKIP and ".ipynb_checkpoints" not in p.parts and "scratch" not in p.parts for cc in code_cells(p)]
rows = []
for gene, cat in SUPP_4A[["target", "source"]].itertuples(index=False):
    pat = re.compile(r"[\"']" + re.escape(gene) + r"[\"']")
    used_by = [label for label, text in code if pat.search(text)]
    rows.append({"gene": gene, "Supp Table 4a category": cat, "in the panel above": gene in panel,
                 "collagen panel (Fig 5e)": gene in COLLAGEN_PANEL, "in Visium gene space": gene in VAR_NAMES,
                 "n analysis files": len({u.split("[")[0] for u in used_by}), "used by": "; ".join(used_by)})
supp4a_use = pd.DataFrame(rows)
print(f"Supp Table 4a: {len(SUPP_4A)} genes; code units searched: {len(code)}")
for g in CHECK_GENES:
    print(f"  {g}: {supp4a_use.loc[supp4a_use['gene'] == g, 'used by'].iloc[0] or 'used by no analysis'}")
print(f"Genes used by no analysis: {supp4a_use.loc[supp4a_use['n analysis files'] == 0, 'gene'].tolist() or 'none'}")
with pd.option_context("display.max_colwidth", 120):
    display(supp4a_use.set_index("gene"))
output.save_table(OUT, "supp_table_4a_gene_use", supp4a_use, panel="Supp Table 4a", index=False)

In [ ]:
# Software versions
print(output.versions(["numpy", "pandas", "scipy", "statsmodels", "matplotlib", "scanpy", "anndata"]).to_string(index=False))